# Two-Dimensional Spline Curves
This code plots a two-dimensional curve expressed in terms of its curvilinear abscissa as ${\mathbf{f}}:{\mathbb{R}}\rightarrow{\mathbb{R}}^{2}.$ The coordinates are Cartesian and made of random cubic splines, with knots being indicated by small black dots. The curve pieces are colored either in marroon or green, according to the sign of its curvature. At the leading edge of the curve, we add its osculating circle, which is locally tangent to the curve and shares its radius of curvature; we display it in lime color when it is on the left of the curve, and in red color when it is on the right. In the margins of the plot, one can find in yellow a representation of the horizontal and vertical components of ${\mathbf{f}}.$ Finally, the large pink dots are those places of the curve where the radius of the osculating circle is smaller than some threshold.

In [ ]:
# Load the required libraries
import ipywidgets as widgets
from IPython.display import display
import math
import matplotlib.patches as patches
from matplotlib.path import Path
import matplotlib.pyplot as plt
import numpy as np

import splinekit as sk # This library

# Setup
support = 15 # Length of the displayed splines
oversampling = 10 # Samples per unit length
hertz = 12 # Refresh rate
kink_initial_radius = 1 / 50 # Initial minimal radius below which the curve is said to have a kink

# Control values
r0 = kink_initial_radius # Adjust the kink threshold
oc = True # Show the osculating circle
kink = True # Show the kinks
knots = True # Show the knots
comp = True # Show the components of the coordinate
slomo = False # Throttle down the refresh rate

# Initialize the generator of random numbers
rng = np.random.default_rng()

# Mathematical constants
w = np.array(
    [
        [
            sk.b_spline(1 + t / oversampling - k, 3)
            for k in range(4) # Support of a cubic spline
        ]
        for t in range(oversampling) # One set of weights per shift
    ],
    dtype = float
) # Precomputed B-spline weights
dw = np.array(
    [
        [
            sk.grad_b_spline(1 + t / oversampling - k, 3)
            for k in range(4) # Support of a cubic spline
        ]
        for t in range(oversampling) # One set of weights per shift
    ],
    dtype = float
) # Precomputed B-spline gradient weights
ddw = np.array(
    [
        [
            sk.diff_b_spline(
                1 + t / oversampling - k,
                degree = 3,
                differentiation_order = 2
            )
            for k in range(4) # Support of a cubic spline
        ]
        for t in range(oversampling) # One set of weights per shift
    ],
    dtype = float
) # Precomputed B-spline second-derivative weights

# Initial variables
# Initial continuously defined horizontal component of the spline coordinate
s1 = sk.PeriodicSpline1D.from_spline_coeff(rng.standard_normal(support + 2), degree = 3)
# Initial continuously defined vertical component of the spline coordinate
s2 = sk.PeriodicSpline1D.from_spline_coeff(rng.standard_normal(support + 2), degree = 3)
# Initial samples of the horizontal and vertical components
x1 = s1.get_samples(0, support_length = support, oversampling = oversampling)
x2 = s2.get_samples(0, support_length = support, oversampling = oversampling)
# Initial samples of the gradient of the horizontal and vertical components
dx1 = s1.gradient().get_samples(0, support_length = support, oversampling = oversampling)
dx2 = s2.gradient().get_samples(0, support_length = support, oversampling = oversampling)
# Initial samples of the second-order derivative of the horizontal and vertical components
ddx1 = s1.differentiated(2).get_samples(0, support_length = support, oversampling = oversampling)
ddx2 = s2.differentiated(2).get_samples(0, support_length = support, oversampling = oversampling)
# Initial knots
isknot = np.array([0 == k % oversampling for k in range(x1.size)], dtype = bool)
# Precomputed running domain of the horizontal and vertical components
hk = np.linspace(start = -2.0, stop = -1.0, num = 3 * oversampling)
vk = np.linspace(start = 1.0, stop = 2.0, num = 3 * oversampling)
# Radius of the osculating circle
def osculating_signed_radius (
    k
):
    numerator = math.sqrt(dx1[k] ** 2 + dx2[k] ** 2) ** 3
    denominator = dx1[k] * ddx2[k] - ddx1[k] * dx2[k]
    if math.isclose(
        0,
        denominator,
        rel_tol = math.sqrt(math.ulp(1.0)),
        abs_tol = math.sqrt(math.ulp(1.0))
    ):
        if math.isclose(
            0,
            numerator,
            rel_tol = math.sqrt(math.ulp(1.0)),
            abs_tol = math.sqrt(math.ulp(1.0))
        ):
            return float("nan")
        return float("inf") if 0 < denominator else -float("inf")
    return numerator / denominator
# Initial radiuses of the osculating circles
r = np.array([osculating_signed_radius(k) for k in range(x1.size)], dtype = float)
# Initial trailing spline coefficients
c1 = np.array(s1.spline_coeff[support - 2 : support + 2], dtype = float)
c2 = np.array(s2.spline_coeff[support - 2 : support + 2], dtype = float)

# Layout of the plot
(fig, ax) = plt.subplots()
ax.set_aspect(1)
ax.spines[:].set_color("lightgray")
plt.tick_params(bottom = False, labelbottom = False, left = False, labelleft = False)
hdisplay = display("", display_id = True)

# Display one frame
def plot_spline (
    t
):
    global x1, x2, dx1, dx2, ddx1, ddx2, isknot, r, c1, c2
    if 0 == t:
        isknot[0] = True
        # Innovation, create new trailing coefficients
        c1[0] = rng.standard_normal()
        c1 = np.roll(c1, -1)
        c2[0] = rng.standard_normal()
        c2 = np.roll(c2, -1)
    else:
        isknot[0] = False
    x1[0] = w[t] @ c1 # New horizontal component trailing sample
    dx1[0] = dw[t] @ c1 # Horizontal gradient
    ddx1[0] = ddw[t] @ c1 # Horizontal second-order derivative
    x2[0] = w[t] @ c2 # New vertical component trailing sample
    dx2[0] = dw[t] @ c2 # Vertical gradient
    ddx2[0] = ddw[t] @ c2 # Vertical second-order derivative
    r[0] = osculating_signed_radius(0) # Radius of the osculating circle
    # Center of the osculating circle
    o1 = float("nan")
    o2 = float("nan")
    if math.isfinite(r[0]):
        o1 = x1[0] - r[0] * dx2[0] / math.sqrt(dx1[0] ** 2 + dx2[0] ** 2)
        o2 = x2[0] + r[0] * dx1[0] / math.sqrt(dx1[0] ** 2 + dx2[0] ** 2)
    # Running window
    x1 = np.roll(x1, -1)
    x2 = np.roll(x2, -1)
    dx1 = np.roll(dx1, -1)
    dx2 = np.roll(dx2, -1)
    ddx1 = np.roll(ddx1, -1)
    ddx2 = np.roll(ddx2, -1)
    isknot = np.roll(isknot, -1)
    r = np.roll(r, -1)
    # New graph
    ax.clear()
    if comp: # Show components
        # Horizontal component
        h = np.flip(x1[ : -3 * oversampling - 1 : -1])
        ax.add_patch(patches.PathPatch(Path(np.transpose([h, hk])), fill = False, color = "gold"))
        # Vertical component
        v = x2[ : -3 * oversampling - 1 : -1]
        ax.add_patch(patches.PathPatch(Path(np.transpose([vk, v])), fill = False, color = "gold"))
        ax.add_patch(patches.PathPatch(
            Path([(x1[-1], x2[-1]), (h[-1], hk[-1]), (x1[-1], x2[-1]), (vk[0], v[0])]),
            fill = False,
            color = "k",
            linewidth = 0.25
        )) # Links between the head of the curve and the components
    # Spline curve
    x = np.transpose([x1, x2])
    k = 0
    while k < len(r): # Stretches of radiuses of identical sign
        q = k + 1
        while q < len(r) and 0 < r[k] * r[q]:
            q += 1
        ax.add_patch(
            patches.PathPatch(
                Path(x[k : q]),
                color = "maroon" if r[k] < 0 else "green",
                fill = False
            )
        )
        if q < len(r):
            ax.add_patch(patches.PathPatch(Path(x[q - 1 : q + 1]), color = "hotpink", fill = False))
        k = q
    if oc: # Osculating circle
        ax.add_patch(
            patches.Circle(
                (o1, o2),
                r[-1],
                fill = False,
                color = "r" if r[-1] < 0 else "lime",
                linewidth = 0.5
            )
        )
    if kink: # Kinks
        for k in range(r.size):
            if abs(r[k]) < r0:
                ax.add_patch(patches.Circle((x1[k], x2[k]), 0.05, color = "pink"))
    if knots: # Knots
        for k in range(isknot.size):
            if isknot[k]:
                ax.add_patch(patches.Circle((x1[k], x2[k]), 0.0125, color = "k"))
    # Do plot the graph update
    ax.set_xlim(-2, 2)
    ax.set_ylim(-2, 2)
    plt.close(fig)
    hdisplay.update(fig)

# Allow for the selection of the largest radius of a kink
r0_slider = widgets.FloatSlider(
    min = 0,
    max = 0.1,
    value = r0,
    step = 0.001,
    readout_format = ".3f"
)
def on_r0_change (
    change
):
    global r0
    r0 = r0_slider.value
r0_slider.observe(on_r0_change, names = "value")

# Display of the oscillating circle
oc_checkbox = widgets.Checkbox(value = oc, description = "Show Osculating Circle")
def on_oc_change (
    value
):
    global oc
    oc = oc_checkbox.value
oc_checkbox.observe(on_oc_change, names = "value")

# Display of the kinks
kink_checkbox = widgets.Checkbox(value = kink, description = "Show Kinks")
def on_kink_change (
    value
):
    global kink
    kink = kink_checkbox.value
kink_checkbox.observe(on_kink_change, names = "value")

# Display of the knots
knots_checkbox = widgets.Checkbox(value = knots, description = "Show Knots")
def on_knots_change (
    value
):
    global knots
    knots = knots_checkbox.value
knots_checkbox.observe(on_knots_change, names = "value")

# Display of the components of the coordinate
comp_checkbox = widgets.Checkbox(value = comp, description = "Show Curve Components")
def on_comp_change (
    value
):
    global comp
    comp = comp_checkbox.value
comp_checkbox.observe(on_comp_change, names = "value")

# Slow motion
slomo_checkbox = widgets.Checkbox(value = slomo, description = "Slow Motion")
def on_slomo_changed (
    change
):
    global slomo, stepper_widget
    slomo = slomo_checkbox.value
    stepper_widget.interval = 1000 if slomo else 1000 / hertz
slomo_checkbox.observe(on_slomo_changed, names = "value")

# Stepper
stepper_widget = widgets.Play(
    value = 0,
    min = 0,
    max = oversampling - 1,
    step = 1,
    interval = 1000 / hertz,
    disabled = False,
    repeat = True,
    show_repeat = False
)
def on_did_step (
    change
):
    plot_spline(stepper_widget.value)
stepper_widget.observe(on_did_step, names = "value")

# Initial figure
plot_spline(0)

# Show controls
widgets.VBox([
    widgets.HBox([widgets.Label(value='Max Kink Radius'), r0_slider]),
    oc_checkbox,
    kink_checkbox,
    knots_checkbox,
    comp_checkbox,
    slomo_checkbox,
    widgets.HBox([widgets.Label(value='Play/Pause:'), stepper_widget])
])
